# TP IA 2025 - Preparación y Optimización
## Parte 1: Predicción de alcohol_freq, Clustering y Algoritmo Genético

Este notebook realiza:
1. Predicción de valores faltantes en alcohol_freq
2. Clustering sustractivo para identificar grupos
3. Algoritmo genético para optimizar arquitectura de red (SIN límite temporal)
4. Guarda la mejor configuración encontrada

**OPTIMIZADO PARA WSL/Ubuntu:**
- Población: 30 (reducida para memoria limitada)
- Generaciones: 10 por ejecución
- Checkpoints automáticos después de cada generación
- Puede reanudar si se interrumpe
- Limpieza agresiva de memoria

**Tiempo estimado**: 1-2 horas por ejecución de 10 generaciones

**Si WSL se cae:**
1. Reinicia WSL
2. Vuelve a ejecutar la celda del AG
3. Se reanudará automáticamente desde el último checkpoint

**Archivos de checkpoint:**
- `ag_checkpoint.pkl` - Estado completo del AG
- `best_model_config_temp.json` - Mejor configuración actual

In [ ]:
# Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
from datetime import datetime
import warnings
import gc
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.spatial import distance_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")
print(f"Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Semillas para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

sns.set(style='whitegrid', palette='muted', font_scale=1.2)

## 1. Carga de Datos

In [ ]:
# Cargar dataset
dirr = "/home/pedro_dev/Pedro/IA/competencia_FF/19-IA2025 medical_insurance.csv"
df = pd.read_csv(dirr)

print(f"Dataset: {df.shape[0]} registros, {df.shape[1]} columnas")
print(f"\nValores faltantes por columna:")
missing = df.isnull().sum()
print(missing[missing > 0])

# Convertir categóricas
categorical_cols = ['sex', 'region', 'urban_rural', 'education', 'marital_status', 
                   'employment_status', 'smoker', 'alcohol_freq', 'plan_type', 'network_tier']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

## 2. Predicción de alcohol_freq
### Red: [1024, 512, 256] con Dropout para evitar overfitting

In [ ]:
print("="*80)
print("PREDICCIÓN DE ALCOHOL_FREQ")
print("="*80)

# Analizar alcohol_freq
print(f"\nValores faltantes: {df['alcohol_freq'].isna().sum()}")
print(f"Valores válidos: {df['alcohol_freq'].notna().sum()}")
print(f"\nCategorías: {sorted(df['alcohol_freq'].dropna().unique())}")
print(f"\nDistribución:")
print(df['alcohol_freq'].value_counts().sort_index())

In [ ]:
# Separar datos válidos vs faltantes
df_valid = df[df['alcohol_freq'].notna()].copy()
df_missing = df[df['alcohol_freq'].isna()].copy()

# Codificar target
label_encoder = LabelEncoder()
y_alcohol = label_encoder.fit_transform(df_valid['alcohol_freq'])
n_classes = len(label_encoder.classes_)

print(f"\nDatos válidos: {len(df_valid)}")
print(f"Datos faltantes: {len(df_missing)}")
print(f"Clases: {label_encoder.classes_}")
print(f"Número de clases: {n_classes}")

In [ ]:
# Preparar features INCLUYENDO annual_medical_cost
# Solo excluir person_id y alcohol_freq
exclude_for_alcohol = ['person_id', 'alcohol_freq']

df_valid_features = df_valid.drop(columns=exclude_for_alcohol, errors='ignore')

categorical_cols_pred = ['sex', 'region', 'urban_rural', 'education', 'marital_status', 
                         'employment_status', 'smoker', 'plan_type', 'network_tier']

X_alcohol = pd.get_dummies(df_valid_features, 
                           columns=[c for c in categorical_cols_pred if c in df_valid_features.columns], 
                           drop_first=True)
X_alcohol = X_alcohol.fillna(X_alcohol.median())

print(f"\nFeatures (con annual_medical_cost): {X_alcohol.shape[1]} columnas")

In [ ]:
# Split 80/20
X_alc_train, X_alc_test, y_alc_train, y_alc_test = train_test_split(
    X_alcohol, y_alcohol, test_size=0.20, random_state=42, stratify=y_alcohol
)

# Normalizar
scaler_alcohol = StandardScaler()
X_alc_train_scaled = scaler_alcohol.fit_transform(X_alc_train)
X_alc_test_scaled = scaler_alcohol.transform(X_alc_test)

# One-hot encoding del target
y_alc_train_cat = to_categorical(y_alc_train, num_classes=n_classes)
y_alc_test_cat = to_categorical(y_alc_test, num_classes=n_classes)

print(f"Train: {len(X_alc_train)} | Test: {len(X_alc_test)}")

In [ ]:
# Modelo con Dropout y Regularización para evitar overfitting
model_alcohol = models.Sequential([
    layers.Input(shape=(X_alc_train_scaled.shape[1],)),
    
    # Layer 1: 1024 neuronas + Dropout
    layers.Dense(1024, activation='relu', 
                kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    
    # Layer 2: 512 neuronas + Dropout
    layers.Dense(512, activation='relu',
                kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    
    # Layer 3: 256 neuronas + Dropout
    layers.Dense(256, activation='relu',
                kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    
    # Output
    layers.Dense(n_classes, activation='softmax')
])

model_alcohol.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModelo creado:")
model_alcohol.summary()

In [ ]:
# Entrenar con Early Stopping más estricto
print("\nEntrenando modelo...")

early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True, verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1
)

history_alcohol = model_alcohol.fit(
    X_alc_train_scaled, y_alc_train_cat,
    validation_data=(X_alc_test_scaled, y_alc_test_cat),
    epochs=100,  # Más epochs pero con early stopping
    batch_size=128,  # Batch más grande para estabilidad
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Evaluar
y_alc_pred_proba = model_alcohol.predict(X_alc_test_scaled, verbose=0)
y_alc_pred = np.argmax(y_alc_pred_proba, axis=1)

accuracy = accuracy_score(y_alc_test, y_alc_pred)
print(f"\n{'='*80}")
print(f"EVALUACIÓN - Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*80}")
print("\nReporte de clasificación:")
print(classification_report(y_alc_test, y_alc_pred, target_names=label_encoder.classes_))

In [ ]:
# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_alcohol.history['accuracy'], label='Train')
axes[0].plot(history_alcohol.history['val_accuracy'], label='Validation')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy - Predicción alcohol_freq')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_alcohol.history['loss'], label='Train')
axes[1].plot(history_alcohol.history['val_loss'], label='Validation')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss - Predicción alcohol_freq')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('alcohol_freq_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Predecir valores faltantes
print("\nPrediciendo valores faltantes...")

df_missing_features = df_missing.drop(columns=exclude_for_alcohol, errors='ignore')
X_missing = pd.get_dummies(df_missing_features, 
                          columns=[c for c in categorical_cols_pred if c in df_missing_features.columns],
                          drop_first=True)

# Alinear columnas
for col in X_alcohol.columns:
    if col not in X_missing.columns:
        X_missing[col] = 0
X_missing = X_missing[X_alcohol.columns]
X_missing = X_missing.fillna(X_alcohol.median())

# Predecir
X_missing_scaled = scaler_alcohol.transform(X_missing)
y_missing_proba = model_alcohol.predict(X_missing_scaled, verbose=0)
y_missing_pred = np.argmax(y_missing_proba, axis=1)
alcohol_freq_predicted = label_encoder.inverse_transform(y_missing_pred)

print(f"\nPredicciones: {len(alcohol_freq_predicted)}")
print("\nDistribución predicha:")
unique, counts = np.unique(alcohol_freq_predicted, return_counts=True)
for cat, count in zip(unique, counts):
    print(f"  {cat}: {count} ({100*count/len(alcohol_freq_predicted):.2f}%)")

In [ ]:
# Actualizar dataframe
df_completed = df.copy()
missing_indices = df[df['alcohol_freq'].isna()].index
df_completed.loc[missing_indices, 'alcohol_freq'] = alcohol_freq_predicted
df = df_completed.copy()

print(f"\n✓ Dataset actualizado")
print(f"Valores faltantes restantes: {df['alcohol_freq'].isna().sum()}")

## 3. Preprocesamiento para Modelo Principal
### Excluir: person_id, annual_medical_cost, annual_premium, monthly_premium

In [ ]:
print("\n" + "="*80)
print("PREPROCESAMIENTO PARA MODELO PRINCIPAL")
print("="*80)

# Target y exclusiones
target = 'annual_medical_cost'
exclude_cols = ['person_id', target, 'annual_premium', 'monthly_premium']

features = df.drop(columns=exclude_cols, errors='ignore')
categorical_features = features.select_dtypes(include=['category', 'object']).columns.tolist()
features_encoded = pd.get_dummies(features, columns=categorical_features, drop_first=True)
features_encoded = features_encoded.fillna(features_encoded.median())

X = features_encoded.values
y = df[target].values

print(f"\nFeatures finales: {X.shape[1]} columnas")
print(f"Excluidas: {exclude_cols}")
print(f"Samples: {len(X)}")
print(f"Target rango: [{y.min():.2f}, {y.max():.2f}]")

## 4. Modelo Baseline

In [ ]:
X_train_bl, X_test_bl, y_train_bl, y_test_bl = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model_baseline = LinearRegression()
model_baseline.fit(X_train_bl, y_train_bl)
y_pred_bl = model_baseline.predict(X_test_bl)

mae_baseline = mean_absolute_error(y_test_bl, y_pred_bl)
mse_baseline = mean_squared_error(y_test_bl, y_pred_bl)
r2_baseline = r2_score(y_test_bl, y_pred_bl)

print("\n" + "="*80)
print("MODELO BASELINE - Regresión Lineal")
print("="*80)
print(f"MAE: {mae_baseline:.4f}")
print(f"MSE: {mse_baseline:.4f}")
print(f"RMSE: {np.sqrt(mse_baseline):.4f}")
print(f"R²: {r2_baseline:.4f}")
print("="*80)
print(f"\n*** OBJETIVO: Superar MAE < {mae_baseline:.4f} ***\n")

## 5. Clustering Sustractivo
### Con dispersión y selección aleatoria

In [ ]:
def subclust2(data, Ra, Rb=0, AcceptRatio=0.3, RejectRatio=0.1):
    """
    Clustering sustractivo.
    """
    if Rb == 0:
        Rb = Ra * 1.15
    
    scaler = MinMaxScaler()
    scaler.fit(data)
    ndata = scaler.transform(data)
    
    P = distance_matrix(ndata, ndata)
    alpha = (Ra / 2) ** 2
    P = np.sum(np.exp(-P**2 / alpha), axis=0)
    
    centers = []
    i = np.argmax(P)
    C = ndata[i]
    p = P[i]
    centers = [C]
    
    continuar = True
    restarP = True
    
    while continuar:
        pAnt = p
        if restarP:
            P = P - p * np.array([np.exp(-np.linalg.norm(v - C)**2 / (Rb / 2)**2) for v in ndata])
        restarP = True
        
        i = np.argmax(P)
        C = ndata[i]
        p = P[i]
        
        if p > AcceptRatio * pAnt:
            centers = np.vstack((centers, C))
        elif p < RejectRatio * pAnt:
            continuar = False
        else:
            dr = np.min([np.linalg.norm(v - C) for v in centers])
            if dr / Ra + p / pAnt >= 1:
                centers = np.vstack((centers, C))
            else:
                P[i] = 0
                restarP = False
        
        if not any(v > 0 for v in P):
            continuar = False
    
    distancias = [[np.linalg.norm(p - c) for p in ndata] for c in centers]
    labels = np.argmin(distancias, axis=0)
    centers = scaler.inverse_transform(centers)
    
    return labels, centers

In [ ]:
print("\n" + "="*80)
print("CLUSTERING INCREMENTAL POR BATCHES")
print("="*80)

def incremental_subclust(data, batch_size=10000, Ra=0.5, AcceptRatio=0.4, RejectRatio=0.15):
    """
    Clustering incremental para datasets grandes.
    Procesa en batches y mantiene solo centros entre batches.
    """
    n_samples = len(data)
    n_batches = int(np.ceil(n_samples / batch_size))
    
    print(f"\nDataset: {n_samples} muestras")
    print(f"Batches: {n_batches} de {batch_size} muestras")
    print(f"Radio: {Ra}\n")
    
    all_centers = []
    scaler = MinMaxScaler()
    
    # Ajustar scaler con muestra representativa (ahorra memoria)
    sample_for_scaling = data[::max(1, len(data)//5000)]  # Max 5000 para scaling
    scaler.fit(sample_for_scaling)
    del sample_for_scaling
    gc.collect()
    
    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, n_samples)
        batch_data = data[start_idx:end_idx]
        
        print(f"Procesando batch {batch_idx + 1}/{n_batches} ({len(batch_data)} muestras)...", end='')
        
        # Normalizar batch
        batch_norm = scaler.transform(batch_data)
        
        # Si hay centros previos, incluirlos en el análisis
        if len(all_centers) > 0:
            # Combinar centros previos normalizados con batch actual
            prev_centers_norm = scaler.transform(all_centers)
            combined = np.vstack([batch_norm, prev_centers_norm])
        else:
            combined = batch_norm
        
        # Aplicar clustering solo en el batch combinado (mucho más pequeño)
        labels_combined, centers_combined_norm = subclust2_simple(
            combined, Ra=Ra, AcceptRatio=AcceptRatio, RejectRatio=RejectRatio
        )
        
        # Desnormalizar centros
        centers_batch = scaler.inverse_transform(centers_combined_norm)
        
        # Actualizar lista de centros (eliminar duplicados muy cercanos)
        if len(all_centers) == 0:
            all_centers = centers_batch
        else:
            # Fusionar centros (mantener solo los únicos)
            all_centers = merge_close_centers(all_centers, centers_batch, threshold=Ra*2)
        
        print(f" → {len(centers_batch)} centros encontrados, total acumulado: {len(all_centers)}")
        
        # Liberar memoria
        del batch_data, batch_norm, combined
        if 'prev_centers_norm' in locals():
            del prev_centers_norm
        gc.collect()
    
    print(f"\n✓ Clustering incremental completado: {len(all_centers)} centros finales")
    
    # Asignar todos los datos a los centros finales
    print("Asignando todos los datos a clusters finales...")
    labels_final = assign_to_clusters_batched(data, all_centers, scaler, batch_size=20000)
    
    return labels_final, all_centers

def subclust2_simple(data, Ra, AcceptRatio=0.3, RejectRatio=0.1):
    """Versión simplificada de subclust2 que asume datos ya normalizados."""
    Rb = Ra * 1.15
    ndata = data  # Ya normalizado
    
    # Calcular potencial usando broadcasting optimizado
    P = distance_matrix(ndata, ndata)
    alpha = (Ra / 2) ** 2
    P = np.sum(np.exp(-P**2 / alpha), axis=0)
    
    centers = []
    i = np.argmax(P)
    C = ndata[i]
    p = P[i]
    centers = [C]
    
    continuar = True
    restarP = True
    
    while continuar:
        pAnt = p
        if restarP:
            P = P - p * np.array([np.exp(-np.linalg.norm(v - C)**2 / (Rb / 2)**2) for v in ndata])
        restarP = True
        
        i = np.argmax(P)
        C = ndata[i]
        p = P[i]
        
        if p > AcceptRatio * pAnt:
            centers = np.vstack((centers, C))
        elif p < RejectRatio * pAnt:
            continuar = False
        else:
            dr = np.min([np.linalg.norm(v - C) for v in centers])
            if dr / Ra + p / pAnt >= 1:
                centers = np.vstack((centers, C))
            else:
                P[i] = 0
                restarP = False
        
        if not any(v > 0 for v in P):
            continuar = False
    
    # Asignar labels
    if len(centers.shape) == 1:
        centers = centers.reshape(1, -1)
    
    distancias = [[np.linalg.norm(p - c) for p in ndata] for c in centers]
    labels = np.argmin(distancias, axis=0)
    
    return labels, centers

def merge_close_centers(centers1, centers2, threshold):
    """Fusiona centros que están muy cerca entre sí."""
    all_centers = np.vstack([centers1, centers2])
    keep_indices = []
    
    for i in range(len(all_centers)):
        is_unique = True
        for j in keep_indices:
            dist = np.linalg.norm(all_centers[i] - all_centers[j])
            if dist < threshold:
                is_unique = False
                break
        if is_unique:
            keep_indices.append(i)
    
    return all_centers[keep_indices]

def assign_to_clusters_batched(data, centers, scaler, batch_size=20000):
    """Asigna datos a clusters en batches para ahorrar memoria."""
    n_samples = len(data)
    labels = np.zeros(n_samples, dtype=np.int32)
    centers_norm = scaler.transform(centers)
    
    n_batches = int(np.ceil(n_samples / batch_size))
    
    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, n_samples)
        batch_data = data[start_idx:end_idx]
        batch_norm = scaler.transform(batch_data)
        
        # Calcular distancias solo para este batch
        distances = np.zeros((len(batch_norm), len(centers_norm)))
        for i, center in enumerate(centers_norm):
            diff = batch_norm - center
            distances[:, i] = np.sqrt(np.sum(diff**2, axis=1))
        
        labels[start_idx:end_idx] = np.argmin(distances, axis=1)
        
        del batch_data, batch_norm, distances
        gc.collect()
        
        if (batch_idx + 1) % 5 == 0 or batch_idx == n_batches - 1:
            print(f"  Asignados: {end_idx}/{n_samples}", end='\r')
    
    print(f"  Asignados: {n_samples}/{n_samples} ✓")
    return labels

# Ejecutar clustering incremental
start_time = time.time()
labels, centers = incremental_subclust(X, batch_size=10000, Ra=0.5)
clustering_time = time.time() - start_time

n_clusters = len(centers)
print(f"\nTiempo total: {clustering_time:.2f}s")
print(f"Clusters finales: {n_clusters}")
print(f"\nDistribución:")
unique, counts = np.unique(labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"  Cluster {cluster_id}: {count} ({100*count/len(labels):.2f}%)")

print("="*80)

## 6. Selección de Datos con Dispersión y Random Sampling

In [ ]:
# Seleccionar datos
X_train_repr, X_test_repr, y_train_repr, y_test_repr = select_representative_samples_with_variance(
    X, y, labels, train_ratio=0.30, test_ratio=0.10
)

print(f"\nDatos seleccionados para AG:")
print(f"  Train: {len(X_train_repr)} ({100*len(X_train_repr)/len(X):.2f}%)")
print(f"  Test: {len(X_test_repr)} ({100*len(X_test_repr)/len(X):.2f}%)")

In [ ]:
def select_representative_samples_with_variance(X, y, labels, train_ratio=0.30, test_ratio=0.10):
    """
    Selecciona datos usando:
    - Media como centro del cluster
    - Cálculo de dispersión
    - Selección ALEATORIA (no los más cercanos)
    
    Esto da mayor variación en los datos seleccionados.
    """
    # IMPORTANTE: Convertir a numpy arrays con dtype correcto
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)
    
    X_train_list = []
    X_test_list = []
    y_train_list = []
    y_test_list = []
    
    n_clusters = len(np.unique(labels))
    
    print("\nCálculo de dispersión por cluster:")
    for cluster_id in range(n_clusters):
        cluster_mask = labels == cluster_id
        X_cluster = X[cluster_mask].astype(np.float64)
        y_cluster = y[cluster_mask].astype(np.float64)
        
        if len(X_cluster) == 0:
            continue
        
        # Calcular media (centro real del cluster)
        center_mean = np.mean(X_cluster, axis=0, dtype=np.float64)
        
        # Calcular dispersión usando broadcasting seguro
        diff = X_cluster - center_mean
        distances = np.sqrt(np.sum(diff**2, axis=1))  # Usar sqrt manual en lugar de norm
        dispersion = np.std(distances)
        mean_distance = np.mean(distances)
        
        print(f"  Cluster {cluster_id}: n={len(X_cluster)}, dispersión={dispersion:.4f}, dist_media={mean_distance:.4f}")
        
        # Selección ALEATORIA (para mayor variación)
        n_train = max(1, int(len(X_cluster) * train_ratio))
        n_test = max(1, int(len(X_cluster) * test_ratio))
        
        # Índices aleatorios
        all_indices = np.arange(len(X_cluster))
        np.random.shuffle(all_indices)
        
        train_indices = all_indices[:n_train]
        test_indices = all_indices[n_train:n_train + n_test]
        
        X_train_list.append(X_cluster[train_indices])
        X_test_list.append(X_cluster[test_indices])
        y_train_list.append(y_cluster[train_indices])
        y_test_list.append(y_cluster[test_indices])
    
    X_train_sel = np.vstack(X_train_list)
    X_test_sel = np.vstack(X_test_list)
    y_train_sel = np.concatenate(y_train_list)
    y_test_sel = np.concatenate(y_test_list)
    
    return X_train_sel, X_test_sel, y_train_sel, y_test_sel

print("Función select_representative_samples_with_variance definida")

## 6.5. Funciones de Checkpoint y Optimización de Memoria

In [ ]:
import pickle

def save_checkpoint(population, history, generation, filename='ag_checkpoint.pkl'):
    """Guarda checkpoint del AG."""
    checkpoint = {
        'population': population,
        'history': history,
        'generation': generation
    }
    with open(filename, 'wb') as f:
        pickle.dump(checkpoint, f)
    print(f"✓ Checkpoint guardado: generación {generation}")

def load_checkpoint(filename='ag_checkpoint.pkl'):
    """Carga checkpoint del AG."""
    try:
        with open(filename, 'rb') as f:
            checkpoint = pickle.load(f)
        print(f"✓ Checkpoint cargado: generación {checkpoint['generation']}")
        return checkpoint
    except FileNotFoundError:
        print("No se encontró checkpoint previo, iniciando desde cero")
        return None

def cleanup_memory():
    """Limpieza agresiva de memoria."""
    gc.collect()
    tf.keras.backend.clear_session()

print("Funciones de checkpoint definidas")

## 7. Algoritmo Genético SIN Restricciones Temporales

In [ ]:
class Individual:
    def __init__(self, n_features):
        self.n_features = n_features
        self.n_layers = np.random.randint(1, 5)
        self.nodes_per_layer = [np.random.randint(16, 256) for _ in range(self.n_layers)]
        self.use_batch_norm = np.random.choice([True, False])
        self.batch_norm_momentum = np.random.uniform(0.8, 0.99) if self.use_batch_norm else 0.9
        self.dropout_rates = [np.random.uniform(0.0, 0.5) for _ in range(self.n_layers)]
        self.activation = np.random.choice(['relu', 'tanh', 'elu'])
        self.learning_rate = 10 ** np.random.uniform(-4, -2)
        self.fitness = None
        self.training_time = None
    
    def mutate(self, mutation_rate=0.001):
        if np.random.random() < mutation_rate:
            old_n_layers = self.n_layers
            self.n_layers = np.clip(self.n_layers + np.random.choice([-1, 1]), 1, 4)
            if self.n_layers > old_n_layers:
                self.nodes_per_layer.append(np.random.randint(16, 256))
                self.dropout_rates.append(np.random.uniform(0.0, 0.5))
            elif self.n_layers < old_n_layers:
                self.nodes_per_layer = self.nodes_per_layer[:-1]
                self.dropout_rates = self.dropout_rates[:-1]
        
        for i in range(self.n_layers):
            if np.random.random() < mutation_rate:
                self.nodes_per_layer[i] = np.clip(
                    self.nodes_per_layer[i] + np.random.randint(-20, 21), 16, 256
                )
        
        if np.random.random() < mutation_rate:
            self.use_batch_norm = not self.use_batch_norm
        
        if np.random.random() < mutation_rate and self.use_batch_norm:
            self.batch_norm_momentum = np.clip(
                self.batch_norm_momentum + np.random.uniform(-0.05, 0.05), 0.8, 0.99
            )
        
        for i in range(self.n_layers):
            if np.random.random() < mutation_rate:
                self.dropout_rates[i] = np.clip(
                    self.dropout_rates[i] + np.random.uniform(-0.1, 0.1), 0.0, 0.5
                )
        
        if np.random.random() < mutation_rate:
            self.activation = np.random.choice(['relu', 'tanh', 'elu'])
        
        if np.random.random() < mutation_rate:
            self.learning_rate = np.clip(
                self.learning_rate * np.random.uniform(0.5, 2.0), 0.0001, 0.01
            )

In [ ]:
def create_model(individual, input_dim):
    """
    Crea un modelo de red feedforward basado en los genes del individuo.
    """
    model = models.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    
    # Capas ocultas
    for i in range(individual.n_layers):
        model.add(layers.Dense(individual.nodes_per_layer[i], activation=individual.activation))
        
        if individual.use_batch_norm:
            model.add(layers.BatchNormalization(momentum=individual.batch_norm_momentum))
        
        if individual.dropout_rates[i] > 0.01:
            model.add(layers.Dropout(individual.dropout_rates[i]))
    
    # Capa de salida
    model.add(layers.Dense(1, activation='linear'))
    
    # Compilar
    optimizer = Adam(learning_rate=individual.learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    return model

print("Función create_model definida correctamente")

In [ ]:
def evaluate_individual(individual, X_train, y_train, X_test, y_test, mae_baseline, epochs=20):
    """
    Evalúa individuo con limpieza agresiva de memoria.
    Reducido a 20 epochs para ahorrar memoria y tiempo.
    """
    try:
        start_time = time.time()
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = create_model(individual, X_train.shape[1])
        
        early_stop = callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True
        )
        
        # Batch size más pequeño para WSL
        batch_size = min(32, len(X_train) // 20)
        
        history = model.fit(
            X_train_scaled, y_train,
            validation_data=(X_test_scaled, y_test),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stop],
            verbose=0
        )
        
        y_pred = model.predict(X_test_scaled, verbose=0).flatten()
        mae = mean_absolute_error(y_test, y_pred)
        training_time = time.time() - start_time
        
        if mae >= mae_baseline:
            fitness = 1.0 / mae_baseline
        else:
            fitness = 1.0 / mae
        
        individual.fitness = fitness
        individual.training_time = training_time
        individual.mae = mae
        
        # Limpieza agresiva de memoria
        del model, scaler, X_train_scaled, X_test_scaled, y_pred, history
        cleanup_memory()
        
        return fitness
        
    except Exception as e:
        print(f"Error: {e}")
        individual.fitness = 0.0
        individual.training_time = 0
        individual.mae = float('inf')
        cleanup_memory()
        return 0.0

In [ ]:
def crossover(parent1, parent2):
    child = Individual(parent1.n_features)
    
    if parent1.fitness > parent2.fitness:
        child.n_layers = parent1.n_layers
        child.nodes_per_layer = parent1.nodes_per_layer.copy()
        child.dropout_rates = parent1.dropout_rates.copy()
    else:
        child.n_layers = parent2.n_layers
        child.nodes_per_layer = parent2.nodes_per_layer.copy()
        child.dropout_rates = parent2.dropout_rates.copy()
    
    child.use_batch_norm = np.random.choice([parent1.use_batch_norm, parent2.use_batch_norm])
    child.batch_norm_momentum = np.random.choice([parent1.batch_norm_momentum, parent2.batch_norm_momentum])
    child.activation = np.random.choice([parent1.activation, parent2.activation])
    child.learning_rate = np.random.choice([parent1.learning_rate, parent2.learning_rate])
    
    return child

In [ ]:
def genetic_algorithm(X_train, y_train, X_test, y_test, mae_baseline,
                     population_size=30, elite_percentage=0.10, 
                     mutation_rate=0.001, generations=10, resume_from_checkpoint=True):
    """
    AG optimizado para WSL con:
    - Población reducida (30 en lugar de 100)
    - Generaciones reducidas (10 en lugar de 20)
    - Checkpoints después de cada generación
    - Capacidad de reanudar
    - Limpieza agresiva de memoria
    """
    n_features = X_train.shape[1]
    elite_size = int(population_size * elite_percentage)
    
    # Intentar cargar checkpoint
    checkpoint = None
    if resume_from_checkpoint:
        checkpoint = load_checkpoint()
    
    if checkpoint:
        population = checkpoint['population']
        history = checkpoint['history']
        start_gen = checkpoint['generation'] + 1
        print(f"\nReanudando desde generación {start_gen}")
    else:
        population = [Individual(n_features) for _ in range(population_size)]
        history = {
            'best_fitness': [],
            'avg_fitness': [],
            'best_mae': [],
            'best_individual': []
        }
        start_gen = 0
        print(f"\nIniciando nuevo AG")
    
    print(f"\n{'='*80}")
    print(f"ALGORITMO GENÉTICO OPTIMIZADO PARA WSL")
    print(f"{'='*80}")
    print(f"Población: {population_size} (reducida para WSL)")
    print(f"Generaciones: {generations}")
    print(f"Elite: {elite_size} | Mutación: {mutation_rate}")
    print(f"MAE Baseline: {mae_baseline:.4f}")
    print(f"{'='*80}\n")
    
    ga_start_time = time.time()
    
    for gen in range(start_gen, generations):
        gen_start = time.time()
        print(f"\n--- Generación {gen + 1}/{generations} ---")
        
        # Evaluar población
        for idx, individual in enumerate(population):
            if individual.fitness is None:
                evaluate_individual(individual, X_train, y_train, X_test, y_test, 
                                  mae_baseline, epochs=20)  # Reducido para memoria
                
                if (idx + 1) % 5 == 0:  # Cada 5 en lugar de 10
                    print(f"  Evaluados: {idx + 1}/{population_size}", end='\r')
                    cleanup_memory()  # Limpieza periódica
        
        # Ordenar
        population.sort(key=lambda x: x.fitness, reverse=True)
        
        # Estadísticas
        best = population[0]
        avg_fit = np.mean([ind.fitness for ind in population])
        
        history['best_fitness'].append(best.fitness)
        history['avg_fitness'].append(avg_fit)
        history['best_mae'].append(best.mae)
        history['best_individual'].append(best)
        
        gen_time = time.time() - gen_start
        print(f"\n  MAE: {best.mae:.4f} | Fitness: {best.fitness:.6f} | Tiempo: {gen_time:.1f}s")
        print(f"  {best.n_layers} capas: {best.nodes_per_layer}, {best.activation}")
        
        # GUARDAR CHECKPOINT después de cada generación
        save_checkpoint(population, history, gen, 'ag_checkpoint.pkl')
        
        # Guardar mejor configuración periódicamente
        if (gen + 1) % 2 == 0:  # Cada 2 generaciones
            best_config = {
                'n_layers': best.n_layers,
                'nodes_per_layer': best.nodes_per_layer,
                'use_batch_norm': bool(best.use_batch_norm),
                'batch_norm_momentum': float(best.batch_norm_momentum),
                'dropout_rates': [float(d) for d in best.dropout_rates],
                'activation': best.activation,
                'learning_rate': float(best.learning_rate),
                'mae_validation': float(best.mae),
                'fitness': float(best.fitness),
                'mae_baseline': float(mae_baseline),
                'improvement_percentage': float((mae_baseline - best.mae) / mae_baseline * 100),
                'generation': gen + 1
            }
            with open('best_model_config_temp.json', 'w') as f:
                json.dump(best_config, f, indent=4)
            print(f"  ✓ Config temporal guardada")
        
        # Reproducción
        new_population = population[:elite_size]
        
        while len(new_population) < population_size:
            parent1 = max(np.random.choice(population[:population_size//2], 2), 
                         key=lambda x: x.fitness)
            parent2 = max(np.random.choice(population[:population_size//2], 2), 
                         key=lambda x: x.fitness)
            child = crossover(parent1, parent2)
            child.mutate(mutation_rate)
            new_population.append(child)
        
        population = new_population
        
        # Limpieza de memoria al final de cada generación
        cleanup_memory()
    
    total_time = time.time() - ga_start_time
    
    print(f"\n{'='*80}")
    print(f"AG COMPLETADO")
    print(f"{'='*80}")
    print(f"Tiempo: {total_time/60:.2f} min")
    print(f"Mejor MAE: {history['best_mae'][-1]:.4f}")
    print(f"Mejora: {((mae_baseline - history['best_mae'][-1])/mae_baseline*100):.2f}%")
    print(f"{'='*80}\n")
    
    return history['best_individual'][-1], history

In [ ]:
# Ejecutar AG optimizado para WSL
print("\n" + "="*80)
print("EJECUTANDO AG OPTIMIZADO")
print("="*80)
print("\nNOTA: Si el proceso se interrumpe, vuelve a ejecutar esta celda.")
print("El AG se reanudará automáticamente desde el último checkpoint.\n")

best_individual, ga_history = genetic_algorithm(
    X_train_repr, y_train_repr, X_test_repr, y_test_repr,
    mae_baseline=mae_baseline,
    population_size=30,      # Reducido de 100
    elite_percentage=0.10,
    mutation_rate=0.001,
    generations=10,          # Reducido de 20, pero puedes ejecutar múltiples veces
    resume_from_checkpoint=True  # Se reanuda automáticamente
)

print("\n✓ AG completado o reanudado exitosamente")

In [ ]:
# Visualizar evolución
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].plot(ga_history['best_fitness'], 'b-', linewidth=2, label='Mejor Fitness')
axes[0].plot(ga_history['avg_fitness'], 'r--', linewidth=2, label='Fitness Promedio')
axes[0].set_xlabel('Generación')
axes[0].set_ylabel('Fitness')
axes[0].set_title('Evolución del Fitness')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ga_history['best_mae'], 'g-', linewidth=2, label='Mejor MAE')
axes[1].axhline(y=mae_baseline, color='r', linestyle='--', linewidth=2, label='Baseline')
axes[1].set_xlabel('Generación')
axes[1].set_ylabel('MAE')
axes[1].set_title('Evolución del MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('genetic_algorithm_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Guardar Mejor Configuración

In [ ]:
# Guardar configuración del mejor modelo
best_config = {
    'n_layers': best_individual.n_layers,
    'nodes_per_layer': best_individual.nodes_per_layer,
    'use_batch_norm': best_individual.use_batch_norm,
    'batch_norm_momentum': float(best_individual.batch_norm_momentum),
    'dropout_rates': [float(d) for d in best_individual.dropout_rates],
    'activation': best_individual.activation,
    'learning_rate': float(best_individual.learning_rate),
    'mae_validation': float(best_individual.mae),
    'fitness': float(best_individual.fitness),
    'mae_baseline': float(mae_baseline),
    'improvement_percentage': float((mae_baseline - best_individual.mae) / mae_baseline * 100)
}

with open('best_model_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

print("\n" + "="*80)
print("MEJOR CONFIGURACIÓN GUARDADA")
print("="*80)
print(f"Archivo: best_model_config.json")
print(f"\nConfiguración:")
print(f"  Capas: {best_config['n_layers']}")
print(f"  Nodos: {best_config['nodes_per_layer']}")
print(f"  Batch Norm: {best_config['use_batch_norm']}")
print(f"  Dropout: {best_config['dropout_rates']}")
print(f"  Activación: {best_config['activation']}")
print(f"  Learning Rate: {best_config['learning_rate']:.6f}")
print(f"\nRendimiento:")
print(f"  MAE Validación: {best_config['mae_validation']:.4f}")
print(f"  MAE Baseline: {best_config['mae_baseline']:.4f}")
print(f"  Mejora: {best_config['improvement_percentage']:.2f}%")
print("="*80)

print(f"\n✓ Preparación completada")
print(f"✓ Usar 'best_model_config.json' en el notebook de entrenamiento final")